In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_lg_extfile(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LGId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "LGKey string,"
        #     "CDI string,"
        #     "PartyName string,"
        #     "PartyTaxIdentificationNumber string,"
        #     "Beneficiary string,"
        #     "LGAgreementKey string,"
        #     "LGType string,"
        #     "IssueDate date,"
        #     "ExpiredDate date,"
        #     "LGAmountEur decimal(18,4),"
        #     "LGOriginalCurrency string,"
        #     "LGCollateral string,"
        #     "TotalFeeAmount decimal(18,4),"
        #     "IsPartialCallFlag boolean,"
        #     "TotalClaimAmount decimal(18,4),"
        #     "LGStatus string,"
        #     "FileNameUploaded string,"
        #     "FileUploadedDate timestamp,"
        #     "LoadingDateTime timestamp,"
        #     "LGFeePQ string,"
        #     "DateOfCallPayment date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Lg"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # =============== extfile_lg ===============

        extfile_lg_extra_filters = None

        extfile_lg_extra_filters = None

        if extfile_lg_extra_filters is not None:
            extfile_lg_extra_filters.append(date_filter)
            extfile_lg_final_filters = " AND ".join(extfile_lg_extra_filters)
        else:
            extfile_lg_final_filters = date_filter

        extfile_lg_selected_fields = [
            "DebtorID",
            "LGContract",
            "DebtorName",
            "DebtorVat",
            "Beneficiary",
            "TypeOfLetter",
            "IssuanceDate",
            "ExpiryDate",
            "LGEurAmount",
            "LGOrgAmount",
            "LGCollaterals",
            "TotalFee",
            "TotalOrPartialCall",
            "AmountOfClaim",
            "LGStatus",
            "FileNameUploaded",
            "FileUploadedDate",
            "LoadingDateTime",
            "LGFeePQ",
            "DateOfCallPayment",
            "BusinessDate",
        ]

        extfile_lg = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "extfile_lg",
            extfile_lg_final_filters,
            extfile_lg_selected_fields,
        )

        # =============== LG_df ============

        LGId_primary_key_fields = [
            extfile_lg.DebtorID,
            extfile_lg.LGContract,
            lit("ext"),
        ]

        lg_df = (
            extfile_lg.withColumn(
                "LGId",
                sha2(concat_ws(";", *LGId_primary_key_fields), 256),
            )
            .withColumn(
                "LGKey", concat_ws(";", extfile_lg.DebtorID, extfile_lg.LGContract)
            )
            .withColumn(
                "IsPartialCallFlag",
                when(extfile_lg["TotalOrPartialCall"].isin("Y"), "1").otherwise("0"),
            )
            .withColumn(
                "DateOfCallPayment",
                when(extfile_lg["DateOfCallPayment"].isin("0000-01-01"), None).otherwise(extfile_lg["DateOfCallPayment"]),
            )
        )

        lg_df = lg_df.select(
            lg_df.LGId.cast("string").alias("LGId"),
            lit("extfile").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            lg_df.LGKey.cast("string").alias("LGkey"),
            lg_df.DebtorID.cast("string").alias("CDI"),
            lg_df.DebtorName.cast("string").alias("PartyName"),
            lg_df.DebtorVat.cast("string").alias("PartyTaxIdentificationNumber"),
            lg_df.Beneficiary.cast("string").alias("Beneficiary"),
            lg_df.LGContract.cast("string").alias("LGAgreementKey"),
            lg_df.TypeOfLetter.cast("string").alias("LGType"),
            lg_df.IssuanceDate.cast("date").alias("IssueDate"),
            lg_df.ExpiryDate.cast("date").alias("ExpiredDate"),
            lg_df.LGEurAmount.alias("LGAmountEur"),
            lg_df.LGOrgAmount.cast("string").alias("LGOriginalCurrency"),
            lg_df.LGCollaterals.cast("string").alias("LGCollateral"),
            lg_df.TotalFee.alias("TotalFeeAmount"),
            lg_df.IsPartialCallFlag.cast("boolean").alias("IsPartialCallFlag"),
            lg_df.AmountOfClaim.alias("TotalClaimAmount"),
            lg_df.LGStatus.alias("LGStatus"),
            lg_df.FileNameUploaded.alias("FileNameUploaded"),
            lg_df.FileUploadedDate.alias("FileUploadedDate"),
            lg_df.LoadingDateTime.alias("LoadingDateTime"),
            lg_df.LGFeePQ.cast("string").alias("LGFeePQ"),
            lg_df.DateOfCallPayment.cast("date").alias("DateOfCallPayment"),
            lg_df.BusinessDate.alias("BusinessDate"),
        )

        # ================== Append ==================

        lg_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_lg_extfile(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)